# Agentic and Self-Corrective RAG

This file teaches you RAG architectures where the LLM makes decisions about retrieval.
By the end, you will know how Agentic RAG, Self-RAG, Corrective RAG, and Adaptive RAG work.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [3]:
import json
import numpy as np
from openai import OpenAI

client = OpenAI(api_key = api_key)

In [4]:
EMBED_MODEL = "text-embedding-3-small"
MODEL = "gpt-4o-mini"

**What happened:** We imported libraries and set up the OpenAI client with our model names.

In [5]:
def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL, 
        input=texts
    )
    embeddings = []
    for item in response.data:
        embeddings.append(item.embedding)
    return embeddings

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

**What happened:** `get_embeddings` converts text to vectors. `cosine_similarity` measures how close two vectors are (1.0 = identical, 0.0 = unrelated).

## Knowledge Base

In [6]:
KB = [
    {"text": "HPA scales pods based on CPU metrics. Default interval is 15 seconds.",
     "topic": "scaling"},
    {"text": "VPA adjusts CPU and memory requests. Modes: Off, Initial, Auto.",
     "topic": "scaling"},
    {"text": "KEDA supports event-driven autoscaling from 50+ sources.",
     "topic": "scaling"},
    {"text": "Prometheus scrapes /metrics endpoints. PromQL for queries.",
     "topic": "monitoring"},
]

In [7]:
doc_texts = []
for doc in KB:
    doc_texts.append(doc['text'])

doc_embeddings = get_embeddings(doc_texts)
doc_embeddings = np.array(doc_embeddings, dtype="float32")

print(len(doc_embeddings))

4


In [8]:
def search_kb(query, top_k=3):
    
    query_embedding = get_embeddings([query])[0]
    
    scores = []
    for i in range(len(KB)):
        score = cosine_similarity(
            query_embedding, 
            doc_embeddings[i]
        )
        scores.append((i, score))
    scores.sort(key=lambda x: x[1], reverse=True)

    results = []
    for i, score in scores[:top_k]:
        results.append((KB[i], score))
    return results

**What happened:** We embedded all documents and created a search function. It returns the most similar documents for any query.

## Agentic RAG

**What it does:** The LLM decides whether to search the knowledge base or answer directly. It uses function calling (tools) to search when it needs specific information.

**When to use it:** Multi-step questions where some parts need documents and others do not.

**Steps:**
1. Give the LLM a search tool
2. The LLM decides if it needs to search
3. If it searches, the results go back to the LLM
4. The LLM generates the final answer

In [26]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search the knowledge base",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

**What happened:** We defined a tool called `search_docs` that the LLM can call. The LLM will see this tool description and decide when to use it.

In [35]:
def run_agent_loop(query):

    messages = [
        {
            "role": "system", 
            "content":
            "You have a search tool for K8s docs. "
            "Use it only when you need specific details."
        },
        {
            "role": "user", 
            "content": query
        }
    ]

    searches = 0


    for step in range(3):
        response = client.chat.completions.create(
            model=MODEL, 
            messages=messages,
            tools=tools, 
            tool_choice="auto"
        )
        message = response.choices[0].message

        print(message)


        if not message.tool_calls:
            return message.content, searches
        

        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            search_query = args["query"]


            results = search_kb(search_query, top_k=2)
            print("RESULTS FROM KNOWLEGDE BASE COSINE SIMILARITY:\n", results)


            context_parts = []
            for doc, score in results:
                context_parts.append(doc["text"])

            context = "\n".join(context_parts)

            messages.append(message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": context
            })

            searches += 1
            print(f"Agent searched: {search_query}")


    response = client.chat.completions.create(
        model=MODEL, 
        messages=messages
    )

    answer = response.choices[0].message.content
    return answer, searches

In [37]:
answer, n = run_agent_loop(
    "What is the default HPA check interval?"
)



print(f"Searches: {n}")
print(f"Answer: {answer}")

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Q7128ZjEADi4DcflXn1LfDnS', function=Function(arguments='{"query":"default HPA check interval"}', name='search_docs'), type='function')])
RESULTS FROM KNOWLEGDE BASE COSINE SIMILARITY:
 [({'text': 'HPA scales pods based on CPU metrics. Default interval is 15 seconds.', 'topic': 'scaling'}, np.float64(0.569084107566225)), ({'text': 'VPA adjusts CPU and memory requests. Modes: Off, Initial, Auto.', 'topic': 'scaling'}, np.float64(0.3330986011285422))]
Agent searched: default HPA check interval
ChatCompletionMessage(content='The default Horizontal Pod Autoscaler (HPA) check interval is 15 seconds.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
Searches: 1
Answer: The default Horizontal Pod Autoscaler (HPA) check interval is 15 seconds.


**What happened:** This is the agent loop. When the LLM calls the search tool, we run the search, add the results to the conversation, and let the LLM continue. If the LLM does not call a tool, it returns the answer directly.

In [38]:
answer, n = run_agent_loop("What is Kubernetes?")
print(f"  Searches: {n}")
print(f"  Answer: {answer}")

ChatCompletionMessage(content='Kubernetes, often abbreviated as K8s, is an open-source container orchestration platform designed to automate the deployment, scaling, and management of containerized applications. Originally developed by Google, Kubernetes helps manage clusters of hosts running Linux containers, providing container-centric infrastructure.\n\nKey features of Kubernetes include:\n\n1. **Automated Deployment**: Kubernetes allows developers to define how their application should run, making it easy to automate deployments and rollbacks.\n\n2. **Scaling and Load Balancing**: It can scale applications up or down based on demand and distribute traffic to ensure smooth operation.\n\n3. **Self-Healing**: Kubernetes can automatically replace or reschedule containers that fail, ensuring that the desired state of the application is maintained.\n\n4. **Service Discovery and Load Balancing**: It offers built-in service discovery and load balancing to connect a service with its clients

**What happened:** The agent answered directly without searching because the question is general knowledge.

## Self-RAG

**What it does:** Self-RAG makes three checks: (1) Should I retrieve? (2) Is the context relevant? (3) Is my answer supported by the context?

**When to use it:** When you need quality guarantees on every answer.

**Steps:**
1. Ask the LLM if retrieval is needed
2. If yes, retrieve documents and check if they are relevant
3. Generate an answer, then check if the answer is supported by the documents

In [42]:
def self_rag_check_retrieval(query):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=50, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                "Does this need a documentation search?\n"
                f'Question: "{query}"\n\n'
                'Return JSON: {"needs_retrieval": true, '
                '"reason": "why"}'
                )
            }
        ]
    )

    content = response.choices[0].message.content
    
    print(content)
    print(json.loads(content))
    
    result = json.loads(content)
    return result

**What happened:** This function asks the LLM whether the question needs a document search or can be answered from general knowledge.

In [43]:
def self_rag_check_relevance(query, context):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=50, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                f"Is this context relevant to: {query}\n\n"
                f"Context: {context[:500]}\n\n"
                'Return JSON: {"is_relevant": true, '
                '"relevance_score": 0.9}'
                )
            }
        ]
    )
    
    content = response.choices[0].message.content

    print(content)
    print(json.loads(content))

    result = json.loads(content)
    return result

**What happened:** This function checks whether the retrieved documents are actually relevant to the question.

In [44]:
def self_rag_check_support(context, answer):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=50, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                "Is this answer supported by the context?\n\n"
                f"Context: {context[:500]}\n"
                f"Answer: {answer}\n\n"
                'Return JSON: {"is_supported": true, '
                '"unsupported_claims": []}'
                )
            }
        ]
    )

    content = response.choices[0].message.content

    print(content)
    print(json.loads(content))

    result = json.loads(content)
    return result

**What happened:** This function checks if the generated answer is supported by the context. It flags any claims not found in the documents.

In [46]:
query = "How does HPA work?"


check1 = self_rag_check_retrieval(query)
print(f"Needs retrieval: {check1}")

results = search_kb(query, top_k=3)

print("RESULTS FROM RETRIEVAL\n", results)

context_parts = []
for doc, score in results:
    context_parts.append(doc["text"])
context = "\n".join(context_parts)


check2 = self_rag_check_relevance(query, context)
print(f"Context relevant: {check2}")

{
  "needs_retrieval": true,
  "reason": "The question requires specific technical details about Horizontal Pod Autoscaler (HPA) that may not be covered in general knowledge."
}
{'needs_retrieval': True, 'reason': 'The question requires specific technical details about Horizontal Pod Autoscaler (HPA) that may not be covered in general knowledge.'}
Needs retrieval: {'needs_retrieval': True, 'reason': 'The question requires specific technical details about Horizontal Pod Autoscaler (HPA) that may not be covered in general knowledge.'}
RESULTS FROM RETRIEVAL
 [({'text': 'HPA scales pods based on CPU metrics. Default interval is 15 seconds.', 'topic': 'scaling'}, np.float64(0.4266650220971652)), ({'text': 'VPA adjusts CPU and memory requests. Modes: Off, Initial, Auto.', 'topic': 'scaling'}, np.float64(0.30623273833023124)), ({'text': 'Prometheus scrapes /metrics endpoints. PromQL for queries.', 'topic': 'monitoring'}, np.float64(0.1155815982985866))]
{"is_relevant": true, "relevance_score

**What happened:** We ran the first two Self-RAG checks. The LLM decided whether retrieval was needed and whether the retrieved context was relevant.

In [47]:
response = client.chat.completions.create(
    model=MODEL, 
    max_tokens=200, 
    temperature=0,
    messages=[
        {"role": "system",
         "content": "Answer based on the context."},
        {"role": "user",
         "content": f"Context:\n{context}\n\nQ: {query}"}
    ]
)
answer = response.choices[0].message.content
print(f"Answer: {answer}")

check3 = self_rag_check_support(context, answer)
print(f"Supported: {check3}")

Answer: Horizontal Pod Autoscaler (HPA) works by automatically adjusting the number of pod replicas in a Kubernetes deployment based on observed CPU utilization or other select metrics. It continuously monitors the resource usage of the pods and compares it to the target utilization specified in the HPA configuration.

Here's how HPA operates:

1. **Metrics Collection**: HPA relies on metrics collected from the pods, typically CPU utilization. It uses the Kubernetes Metrics Server to gather these metrics.

2. **Target Utilization**: You define a target CPU utilization percentage in the HPA configuration. For example, you might set a target of 50% CPU utilization.

3. **Scaling Decision**: HPA checks the current CPU usage of the pods at a default interval of 15 seconds. If the average CPU usage across the pods exceeds the target utilization, HPA will increase the number of replicas. Conversely, if the usage falls below the target, it may decrease the number of replicas.

4. **Scaling Ac

**What happened:** We generated an answer and then checked if every claim in the answer is supported by the context.

## Corrective RAG (CRAG)

**What it does:** CRAG evaluates retrieval quality and takes different actions based on the result. Good retrieval uses the documents. Bad retrieval falls back to the LLM's own knowledge. Ambiguous retrieval refines the query and tries again.

**When to use it:** When your knowledge base has gaps and you need graceful fallback.

**Steps:**
1. Retrieve documents
2. Evaluate retrieval quality (good / ambiguous / bad)
3. If good: use documents. If ambiguous: refine query and re-search. If bad: answer without documents

In [56]:
def evaluate_retrieval(query, context):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=50, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                f"Rate how well these docs answer: {query}\n\n"
                f"Documents: {context[:500]}\n\n"
                'Return JSON: {"quality": "good/ambiguous/bad", '
                '"reason": "why"}'
                )
            }
        ]
    )

    content = response.choices[0].message.content

    print(content)
    print(json.loads(content))

    result = json.loads(content)
    return result

**What happened:** This function asks the LLM to rate retrieval quality as good, ambiguous, or bad.

In [57]:
def corrective_rag(query):
    
    results = search_kb(query, top_k=3)
    
    context_parts = []
    for doc, score in results:
        context_parts.append(doc["text"])
    context = "\n".join(context_parts)

    print("CONTEXT:\n", context)

    
    quality = evaluate_retrieval(query, context)
    print(f"  Quality: {quality}")

    if quality["quality"] == "bad":
        prompt = query
    else:
        prompt = f"Context:\n{context}\n\nQ: {query}"

    
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=200, 
        temperature=0,
        messages=[
            {
                "role": "user", 
                "content": prompt
            }
        ]
    )
    answer = response.choices[0].message.content
    return answer

**What happened:** CRAG retrieves documents, evaluates their quality, and either uses them or falls back to the LLM's own knowledge.

In [58]:
print("Good retrieval:")
print(corrective_rag("How does HPA calculate replicas?"))

print("\nBad retrieval:")
print(corrective_rag("What is the best pizza topping?"))

Good retrieval:
CONTEXT:
 HPA scales pods based on CPU metrics. Default interval is 15 seconds.
KEDA supports event-driven autoscaling from 50+ sources.
Prometheus scrapes /metrics endpoints. PromQL for queries.
{
  "quality": "bad",
  "reason": "The documents do not provide any information on how HPA calculates replicas specifically; they only mention scaling based on CPU metrics and other unrelated topics."
}
{'quality': 'bad', 'reason': 'The documents do not provide any information on how HPA calculates replicas specifically; they only mention scaling based on CPU metrics and other unrelated topics.'}
  Quality: {'quality': 'bad', 'reason': 'The documents do not provide any information on how HPA calculates replicas specifically; they only mention scaling based on CPU metrics and other unrelated topics.'}
Horizontal Pod Autoscaler (HPA) in Kubernetes automatically adjusts the number of pod replicas in a deployment or replica set based on observed CPU utilization (or other select met

**What happened:** The first query found good documents and used them. The second query had no relevant documents, so CRAG fell back to the LLM's own knowledge.

## Adaptive RAG

**What it does:** Adaptive RAG classifies each query by type (factual, analytical, conversational, comparison) and picks the best retrieval strategy for that type.

**When to use it:** When your system gets many different kinds of questions.

**Steps:**
1. Classify the query type
2. Route to the right strategy (simple search, multi-query, direct answer, etc.)
3. Generate the answer

In [ ]:
def classify_query(query):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=50, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                "Classify this query type:\n"
                "- factual: needs a specific fact\n"
                "- analytical: needs multiple sources\n"
                "- conversational: no docs needed\n"
                "- comparison: comparing things\n\n"
                f'Query: "{query}"\n\n'
                'Return JSON: {"type": "factual", "reason": "..."}'
                )
            }
        ]
    )

    content = response.choices[0].message.content

    print(content)
    print(json.loads(content))

    result = json.loads(content)
    return result